#  BRONZE (Dados Brutos)
Dados exatamente como chegam da fonte<br>
Sem transformações<br>
Serve como backup histórico<br>

## PROCESSAMENTO DOS DADOS

### IMPORTAÇÃO DAS BIBLIOTECAS

In [9]:
from pathlib import Path
from openpyxl import load_workbook
from pyspark.sql import functions as F
from pyspark.sql import types as T
from spark_utils import get_spark, write_single_csv

spark = get_spark("BronzeLayer")
raw_file = Path("data/raw/dados_credito.xlsx")
if not raw_file.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {raw_file}")

schema = T.StructType([
    T.StructField("CODIGO_CLIENTE", T.LongType(), True),
    T.StructField("UF", T.StringType(), True),
    T.StructField("IDADE", T.IntegerType(), True),
    T.StructField("ESCOLARIDADE", T.StringType(), True),
    T.StructField("ESTADO_CIVIL", T.StringType(), True),
    T.StructField("QT_FILHOS", T.IntegerType(), True),
    T.StructField("CASA_PROPRIA", T.StringType(), True),
    T.StructField("QT_IMOVEIS", T.IntegerType(), True),
    T.StructField("VL_IMOVEIS", T.DoubleType(), True),
    T.StructField("OUTRA_RENDA", T.StringType(), True),
    T.StructField("OUTRA_RENDA_VALOR", T.DoubleType(), True),
    T.StructField("TEMPO_ULTIMO_EMPREGO_MESES", T.IntegerType(), True),
    T.StructField("TRABALHANDO_ATUALMENTE", T.StringType(), True),
    T.StructField("ULTIMO_SALARIO", T.DoubleType(), True),
    T.StructField("QT_CARROS", T.IntegerType(), True),
    T.StructField("VALOR_TABELA_CARROS", T.DoubleType(), True),
    T.StructField("SCORE", T.DoubleType(), True)
])


### CRIAÇÃO DA ESTRUTURA DE PASTAS

In [10]:
for folder in [Path("data/bronze"), Path("data/silver"), Path("data/gold")]:
    folder.mkdir(parents=True, exist_ok=True)
print("Estrutura de diretórios verificada.")


Estrutura de diretórios verificada.


### CARREGAMENTO DOS DADOS

In [11]:
wb = load_workbook(raw_file, data_only=True)
sheet = wb.active
rows_iter = sheet.iter_rows(values_only=True)
headers = [str(value).strip() for value in next(rows_iter)]
records = [dict(zip(headers, row)) for row in rows_iter if any(row)]

blank_markers = {"", "None", "NULL"}

def cast_value(value, data_type):
    if value is None:
        return None
    if isinstance(value, str):
        cleaned = value.strip()
        if cleaned in blank_markers:
            return None
        value = cleaned
    try:
        if isinstance(data_type, T.IntegralType):
            return int(float(value))
        if isinstance(data_type, T.FractionalType):
            return float(value)
    except (ValueError, TypeError):
        return None
    return str(value)

clean_records = []
for record in records:
    sanitized = {}
    for field in schema:
        sanitized[field.name] = cast_value(record.get(field.name), field.dataType)
    clean_records.append(sanitized)

df = spark.createDataFrame(clean_records, schema=schema)
print(f"Dimensão inicial do dataset: ({df.count()}, {len(df.columns)})")
df.printSchema()


Dimensão inicial do dataset: (10476, 17)
root
 |-- CODIGO_CLIENTE: long (nullable = true)
 |-- UF: string (nullable = true)
 |-- IDADE: integer (nullable = true)
 |-- ESCOLARIDADE: string (nullable = true)
 |-- ESTADO_CIVIL: string (nullable = true)
 |-- QT_FILHOS: integer (nullable = true)
 |-- CASA_PROPRIA: string (nullable = true)
 |-- QT_IMOVEIS: integer (nullable = true)
 |-- VL_IMOVEIS: double (nullable = true)
 |-- OUTRA_RENDA: string (nullable = true)
 |-- OUTRA_RENDA_VALOR: double (nullable = true)
 |-- TEMPO_ULTIMO_EMPREGO_MESES: integer (nullable = true)
 |-- TRABALHANDO_ATUALMENTE: string (nullable = true)
 |-- ULTIMO_SALARIO: double (nullable = true)
 |-- QT_CARROS: integer (nullable = true)
 |-- VALOR_TABELA_CARROS: double (nullable = true)
 |-- SCORE: double (nullable = true)



## ANÁLISE DOS DADOS

### INSPEÇÃO INICIAL 

In [12]:
df.describe().show(truncate=False)
null_overview = df.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in df.columns])
null_overview.show(truncate=False)


25/11/10 20:26:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-----+------------------+---------------------+------------+-----------------+------------+------------------+------------------+-----------+------------------+--------------------------+----------------------+-----------------+------------------+-------------------+-----------------+
|summary|CODIGO_CLIENTE    |UF   |IDADE             |ESCOLARIDADE         |ESTADO_CIVIL|QT_FILHOS        |CASA_PROPRIA|QT_IMOVEIS        |VL_IMOVEIS        |OUTRA_RENDA|OUTRA_RENDA_VALOR |TEMPO_ULTIMO_EMPREGO_MESES|TRABALHANDO_ATUALMENTE|ULTIMO_SALARIO   |QT_CARROS         |VALOR_TABELA_CARROS|SCORE            |
+-------+------------------+-----+------------------+---------------------+------------+-----------------+------------+------------------+------------------+-----------+------------------+--------------------------+----------------------+-----------------+------------------+-------------------+-----------------+
|count  |10476             |10476|10476             |10476

## SALVAR NA CAMADA BRONZE

### ADIÇÃO DA INFORMAÇÃO DE CARREGAMENTO DOS DADOS


In [13]:
df = (
    df.withColumn("DATA_UPLOAD", F.current_timestamp())
      .withColumn("ARQUIVO_FONTE", F.lit(raw_file.name))
)


### SALVAR DADOS NA CAMADA BRONZE EM ARQUIVO CSV


In [14]:
bronze_path = "data/bronze/dados_brutos.csv"
write_single_csv(df, bronze_path)
print(f">>> Bronze salvo em: {bronze_path}")
df.show(5, truncate=False)


>>> Bronze salvo em: data/bronze/dados_brutos.csv
+--------------+---+-----+---------------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+----------------+--------------------------+------------------+
|CODIGO_CLIENTE|UF |IDADE|ESCOLARIDADE         |ESTADO_CIVIL|QT_FILHOS|CASA_PROPRIA|QT_IMOVEIS|VL_IMOVEIS|OUTRA_RENDA|OUTRA_RENDA_VALOR|TEMPO_ULTIMO_EMPREGO_MESES|TRABALHANDO_ATUALMENTE|ULTIMO_SALARIO|QT_CARROS|VALOR_TABELA_CARROS|SCORE           |DATA_UPLOAD               |ARQUIVO_FONTE     |
+--------------+---+-----+---------------------+------------+---------+------------+----------+----------+-----------+-----------------+--------------------------+----------------------+--------------+---------+-------------------+----------------+--------------------------+------------------+
|1             |SP |19   |Superior Cursando    |Solteiro    |0   